[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境检查与 config 管线

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
跑通环境，并学会本课**所有 notebook 都复用**的真实模型 config 下载范式——用真实架构数字算参数量、和官方对上。

**本 notebook 你将：**
1. 实现 `load_config(model)`：下载并标准化真实 HF config
2. 从架构数字（L/h/heads/V/I）算参数量
3. 验证你的公式：算出的参数量 = 模型官方公布的规模
4. 看 Pythia 全家桶（160M→12B）作为本课的真实"scaling 实验台"

> 数据：EleutherAI Pythia / GPT-NeoX / GPT-2 的 config.json（HuggingFace, 联网下载）。

## 1 · 全课复用的 config 管线

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)

# 真实模型 -> HuggingFace config.json 链接
MODELS = {
  "gpt2":        "https://huggingface.co/openai-community/gpt2/resolve/main/config.json",
  "gpt2-xl":     "https://huggingface.co/openai-community/gpt2-xl/resolve/main/config.json",
  "pythia-160m": "https://huggingface.co/EleutherAI/pythia-160m/resolve/main/config.json",
  "pythia-410m": "https://huggingface.co/EleutherAI/pythia-410m/resolve/main/config.json",
  "pythia-1.4b": "https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json",
  "pythia-2.8b": "https://huggingface.co/EleutherAI/pythia-2.8b/resolve/main/config.json",
  "pythia-6.9b": "https://huggingface.co/EleutherAI/pythia-6.9b/resolve/main/config.json",
  "pythia-12b":  "https://huggingface.co/EleutherAI/pythia-12b/resolve/main/config.json",
  "gpt-neox-20b":"https://huggingface.co/EleutherAI/gpt-neox-20b/resolve/main/config.json",
}

def load_config(model):
    "下载并缓存 HF config，标准化字段名为 L/h/heads/V/I。"
    path=os.path.join(CACHE, f"{model}.json")
    if not os.path.exists(path):
        urllib.request.urlretrieve(MODELS[model], path)
    c=json.load(open(path))
    g=lambda *ks: next(c[k] for k in ks if k in c)
    return dict(model=model,
        L=g("num_hidden_layers","n_layer"),
        h=g("hidden_size","n_embd"),
        heads=g("num_attention_heads","n_head"),
        V=g("vocab_size"),
        I=c.get("intermediate_size", 4*g("hidden_size","n_embd")))

cfg=load_config("pythia-1.4b")
print("pythia-1.4b 标准化 config:", cfg)

## 2 · 从架构数字算参数量

Transformer 参数量的账本（GPT-NeoX 风格，**untied** 输入/输出 embedding）：

- 两个 embedding: `2·V·h`
- 每层 attention（q,k,v,o 投影 + bias）: `4h² + 4h`
- 每层 MLP（up + down + bias）: `2·h·I + (I+h)`
- 每层 2 个 LayerNorm: `4h`
- 最后一个 norm: `~h`

In [ ]:
def param_count(cfg):
    L,h,V,I = cfg["L"],cfg["h"],cfg["V"],cfg["I"]
    emb = 2*V*h
    per_layer = 4*h*h + 4*h + 2*h*I + (I+h) + 4*h
    return emb + L*per_layer + h

for m in ["pythia-160m","pythia-1.4b","pythia-6.9b","pythia-12b"]:
    p = param_count(load_config(m))
    print(f"{m:12s}  算出 {p/1e6:7.0f}M 参数")

## 3 · 验证：账能对上官方数字

Pythia 的命名规模就是它的总参数量。我们算出的应该非常接近。

In [ ]:
known = {"pythia-160m":162e6, "pythia-1.4b":1.41e9, "pythia-6.9b":6.86e9, "pythia-12b":11.85e9}
for m,k in known.items():
    p=param_count(load_config(m)); err=abs(p-k)/k
    print(f"{m:12s} 算出={p/1e6:7.0f}M  官方≈{k/1e6:7.0f}M  误差={err:.1%}  {'✓' if err<0.03 else '✗'}")
print("\n=> 账能对上官方数字，说明你真正理解了参数从哪来")

---
## ✏️ 练习区

### ✏️ 练习 1：标准化任意 config

实现 `normalize(raw)`：输入一个原始 config dict（字段名可能是 GPT-2 风格 `n_layer/n_embd/n_head`
或 NeoX 风格 `num_hidden_layers/hidden_size/num_attention_heads`），返回统一的 `dict(L,h,heads,V,I)`。
缺 `intermediate_size` 时默认 `4*h`。

In [ ]:
def normalize(raw):
    # TODO: 兼容两套字段名，返回 dict(L,h,heads,V,I)
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测（用真实 GPT-2 与 Pythia 原始 config）——
import json
gpt2_raw = json.load(open(os.path.join(CACHE,"gpt2.json"))) if os.path.exists(os.path.join(CACHE,"gpt2.json")) else None
if gpt2_raw is None:
    urllib.request.urlretrieve(MODELS["gpt2"], os.path.join(CACHE,"gpt2.json"))
    gpt2_raw = json.load(open(os.path.join(CACHE,"gpt2.json")))
n = normalize(gpt2_raw)
assert n["L"]==12 and n["h"]==768 and n["heads"]==12 and n["V"]==50257
assert n["I"]==4*768, "GPT-2 没写 intermediate_size，应默认 4h"
py_raw = json.load(open(os.path.join(CACHE,"pythia-1.4b.json")))
n2 = normalize(py_raw)
assert n2["L"]==24 and n2["h"]==2048 and n2["I"]==8192
print("练习 1 通过 ✓")


### ✏️ 练习 2：参数量公式

实现 `count_params(cfg)`（untied embedding）。要求算出的 pythia-410m 在 350–450M 之间。

In [ ]:
def count_params(cfg):
    # TODO: 2*V*h + L*(4h²+4h + 2hI+(I+h) + 4h) + h
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
p = count_params(load_config("pythia-410m"))
assert 350e6 < p < 460e6, f"pythia-410m 应≈405M，你算出 {p/1e6:.0f}M"
# 12b 应在 11.5-12.5B
assert 11.3e9 < count_params(load_config("pythia-12b")) < 12.5e9
print(f"练习 2 通过 ✓  pythia-410m = {p/1e6:.0f}M")


### ✏️ 练习 3：非 embedding 参数占比

实现 `non_embedding_fraction(cfg)`：返回"非 embedding 参数 / 总参数"。
观察：模型越大，这个比例越接近 1（embedding 摊薄）。

In [ ]:
def non_embedding_fraction(cfg):
    # TODO: (总参数 - 2*V*h) / 总参数
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
f_small = non_embedding_fraction(load_config("pythia-160m"))
f_big   = non_embedding_fraction(load_config("pythia-12b"))
assert 0 < f_small < f_big < 1, "大模型 embedding 占比应更小"
assert f_small < 0.6, "160m 中 embedding 占很大一块"
assert f_big > 0.95, "12b 中 embedding 几乎可忽略"
print(f"练习 3 通过 ✓  非embedding占比: 160m={f_small:.2f}  12b={f_big:.2f}")


---
## 📖 参考答案

In [ ]:
# 练习 1
def normalize(raw):
    g=lambda *ks: next(raw[k] for k in ks if k in raw)
    h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"), h=h,
                heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"), I=raw.get("intermediate_size", 4*h))
print("练习 1 ✓")

In [ ]:
# 练习 2
def count_params(cfg):
    L,h,V,I=cfg["L"],cfg["h"],cfg["V"],cfg["I"]
    return 2*V*h + L*(4*h*h+4*h + 2*h*I+(I+h) + 4*h) + h
print("练习 2 ✓")

In [ ]:
# 练习 3
def non_embedding_fraction(cfg):
    tot=count_params(cfg); emb=2*cfg["V"]*cfg["h"]
    return (tot-emb)/tot
print("练习 3 ✓ —— embedding 在小模型里占大头，这解释了为什么小模型'每参数算力'低")